In [ ]:
import numpy as np
import pandas as pd


# -----------------------------
# Scenario settings
# -----------------------------
Esize = 170
Psize = 0
Ssize = 300
Bsize = 0

electrolyzer_efficiency = 62.7  # %
Csize = 8  # kg H2/hour
Initial_storage_status = 0

sell_storage_threshold = 0.95 * Ssize
Freedayrange = [3]
H2prices = [1.5]

simulation_years = 14
Lifetime = 30
Discount_Rate = 0.05


# -----------------------------
# Load data
# -----------------------------
PV = pd.read_csv("PVhourlyHydrogen500.csv", delimiter=";")
PriceAllYears = pd.read_csv("PriceAllYears.csv", delimiter=";")


# -----------------------------
# Helper values
# -----------------------------
discount_factor = sum(
    1 / (1 + Discount_Rate) ** t
    for t in range(1, Lifetime + 1)
)

economic_df = pd.DataFrame()


# -----------------------------
# Main loop
# -----------------------------
for sell_hydrogen_price in H2prices:
    for sell_free_days in Freedayrange:

        battery_status = [0]
        storage_status = [Initial_storage_status]

        excess_energy = []
        electrolyzer_production = []
        pv_to_electrolyzer = []
        grid_to_electrolyzer = []
        compressor_flow_list = []

        hydrogen_sold = []
        h2_to_consumption_list = []
        h2_deficit_list = []
        sell_flag = []

        Simh = PV[
            ["Year", "Month", "Day", "Hour", "Production (Wh)", "Consumption (kWh H2"]
        ].copy()

        Simh["PV production (kWh)"] = Simh["Production (Wh)"] / 1000 * Psize
        Simh["Consumption (kg H2)"] = Simh["Consumption (kWh H2"] / 33.3

        # Mark hours where there is no hydrogen demand in the next N days
        lookahead_hours = sell_free_days * 24

        future_consumption = (
            Simh["Consumption (kg H2)"]
            .iloc[::-1]
            .rolling(lookahead_hours + 1, min_periods=1)
            .sum()
            .iloc[::-1]
        )

        Simh["Sell window"] = (future_consumption <= 1e-9).astype(int)

        Simh = Simh.merge(
            PriceAllYears,
            on=["Year", "Month", "Day", "Hour"],
            how="left"
        )

        price_threshold = Simh["N4 (EUR/kWh)"].quantile(0.8)
        storage_threshold = Ssize

        for i in range(len(Simh)):
            prev_battery = battery_status[-1]
            prev_storage = storage_status[-1]

            pv_production = Simh.loc[i, "PV production (kWh)"]
            current_price = Simh.loc[i, "N4 (EUR/kWh)"]
            consumption = Simh.loc[i, "Consumption (kg H2)"]

            # PV to electrolyzer
            compressor_at_capacity = i > 0 and compressor_flow_list[-1] >= Csize
            storage_nearly_full = prev_storage >= Ssize - Csize

            if compressor_at_capacity or storage_nearly_full:
                pv_to_el = 0
                excess_e = pv_production
            else:
                pv_to_el = min(pv_production, Esize)
                excess_e = max(0, pv_production - pv_to_el)

            # Battery
            charged_battery = min(Bsize, max(0, prev_battery + excess_e))
            battery_charge_added = charged_battery - prev_battery
            excess_e = max(0, excess_e - battery_charge_added)

            remaining_el_capacity = Esize - pv_to_el
            battery_used = 0

            if remaining_el_capacity > 0 and prev_storage < Ssize - Csize:
                battery_used = min(charged_battery, remaining_el_capacity)

            new_battery = max(0, charged_battery - battery_used)
            remaining_el_capacity -= battery_used

            # Grid to electrolyzer
            grid_input = 0

            if (
                remaining_el_capacity > 0
                and current_price < price_threshold
                and prev_storage < min(storage_threshold, Ssize - Csize)
            ):
                grid_input = remaining_el_capacity

            electrolyzer_energy = min(
                pv_to_el + battery_used + grid_input,
                Esize
            )

            # Hydrogen production
            h2_produced = electrolyzer_energy * electrolyzer_efficiency / 100 / 33.3
            compressor_flow = min(h2_produced, Csize)

            # Hydrogen storage and consumption
            available_h2 = prev_storage + compressor_flow
            h2_to_consumption = min(consumption, available_h2)
            h2_deficit = consumption - h2_to_consumption

            storage_after_consumption = min(
                Ssize,
                max(0, available_h2 - consumption)
            )

            # Hydrogen selling
            can_sell = (
                Ssize > 0
                and Simh.loc[i, "Sell window"] == 1
                and storage_after_consumption >= sell_storage_threshold
            )

            if can_sell:
                h2_sold = storage_after_consumption
                sell_event = 1
            else:
                h2_sold = 0
                sell_event = 0

            new_storage = min(
                Ssize,
                max(0, storage_after_consumption - h2_sold)
            )

            battery_status.append(new_battery)
            storage_status.append(new_storage)

            excess_energy.append(excess_e)
            electrolyzer_production.append(electrolyzer_energy)
            pv_to_electrolyzer.append(pv_to_el)
            grid_to_electrolyzer.append(grid_input)
            compressor_flow_list.append(compressor_flow)

            hydrogen_sold.append(h2_sold)
            h2_to_consumption_list.append(h2_to_consumption)
            h2_deficit_list.append(h2_deficit)
            sell_flag.append(sell_event)

        # -----------------------------
        # Add simulation results
        # -----------------------------
        Simh["Battery status (kWh)"] = battery_status[1:]
        Simh["Excess e (kWh)"] = excess_energy
        Simh["Electrolyzer production (kWh)"] = electrolyzer_production
        Simh["PV to Electrolyzer (kWh)"] = pv_to_electrolyzer
        Simh["Grid to Electrolyzer (kWh)"] = grid_to_electrolyzer

        Simh["Hydrogen produced (kg)"] = (
            Simh["Electrolyzer production (kWh)"]
            * electrolyzer_efficiency
            / 100
            / 33.3
        )

        Simh["Compressor flow (kg H2)"] = compressor_flow_list
        Simh["H2 storage (kg H2)"] = storage_status[1:]
        Simh["Hydrogen sold (kg)"] = hydrogen_sold
        Simh["H2 to consumption (kg H2)"] = h2_to_consumption_list
        Simh["H2def (kg H2)"] = h2_deficit_list
        Simh["Selling event"] = sell_flag

        Simh.to_csv("Simh.csv", index=False, sep=";")

        # -----------------------------
        # Economic calculations
        # -----------------------------
        H2_Storage_Capex = 1100 * Ssize
        Annual_H2_Storage_Opex = 0.01 * H2_Storage_Capex

        Battery_Capex = 800 * Bsize
        Annual_Battery_Opex = 0.02 * Battery_Capex

        Compressor_Capex = 240000
        Annual_Compressor_Opex = 0.005 * Compressor_Capex

        PV_Capex = 900 * Psize
        Annual_PV_Opex = 0.015 * PV_Capex

        Electrolyzer_Capex_unit = 1.1195 * (
            585.85 + Esize ** 0.622 * 9485.2 / Esize
        )
        Electrolyzer_CAPEX = Esize * Electrolyzer_Capex_unit
        Annual_Electrolyzer_Opex = 0.02 * Electrolyzer_CAPEX
        Electrolyzer_Replacement_Cost = 0.35 * Electrolyzer_CAPEX

        Electrolyzer_Water_Consumption = 10  # L/kg H2
        Water_Cost = 0.00053  # EUR/L

        Monthly_N4_Fixed_Fee = 42
        Monthly_N4_Power_Fee = 4.1 * Esize

        Annual_N4_Electricity_Fixed_Cost = 12 * (
            Monthly_N4_Fixed_Fee + Monthly_N4_Power_Fee
        )

        Total_CAPEX = (
            PV_Capex
            + Battery_Capex
            + Electrolyzer_CAPEX
            + H2_Storage_Capex
            + Compressor_Capex
        )

        Total_annual_OPEX_fix = (
            Annual_Electrolyzer_Opex
            + Annual_PV_Opex
            + Annual_Compressor_Opex
            + Annual_Battery_Opex
            + Annual_H2_Storage_Opex
            + Annual_N4_Electricity_Fixed_Cost
        )

        Total_Discounted_OPEX_fix = Total_annual_OPEX_fix * discount_factor

        Total_Discounted_Replacement_Cost = (
            Electrolyzer_Replacement_Cost / (1 + Discount_Rate) ** 10
            + Electrolyzer_Replacement_Cost / (1 + Discount_Rate) ** 20
        )

        Simh["Electricity Cost (EUR)"] = (
            Simh["Grid to Electrolyzer (kWh)"] * Simh["N4 (EUR/kWh)"]
        )

        Simh["Electricity Income (EUR)"] = (
            Simh["Excess e (kWh)"] * Simh["Income (EUR/kWh)"]
        )

        Simh["Hydrogen Sale Income (EUR)"] = (
            Simh["Hydrogen sold (kg)"] * sell_hydrogen_price
        )

        Annual_Electricity_Cost = Simh["Electricity Cost (EUR)"].sum() / simulation_years
        Annual_Electricity_Income = Simh["Electricity Income (EUR)"].sum() / simulation_years
        Annual_Hydrogen_Income = Simh["Hydrogen Sale Income (EUR)"].sum() / simulation_years

        Total_Discounted_Electricity_Cost = Annual_Electricity_Cost * discount_factor
        Total_Discounted_Electricity_Income = Annual_Electricity_Income * discount_factor
        Total_Discounted_Hydrogen_Income = Annual_Hydrogen_Income * discount_factor

        Annual_H2_Produced = Simh["Hydrogen produced (kg)"].sum() / simulation_years
        Annual_H2_Consumed = Simh["H2 to consumption (kg H2)"].sum() / simulation_years

        Annual_Water_Cost = (
            Electrolyzer_Water_Consumption
            * Water_Cost
            * Annual_H2_Produced
        )

        Total_Discounted_Water_Cost = Annual_Water_Cost * discount_factor

        Total_H2_Produced = Lifetime * Annual_H2_Produced
        Total_H2_Consumed = Lifetime * Annual_H2_Consumed

        Total_discounted_H2 = Annual_H2_Produced * discount_factor
        Total_discounted_H2_Consumption = Annual_H2_Consumed * discount_factor

        Net_Present_Cost = (
            Total_CAPEX
            + Total_Discounted_OPEX_fix
            + Total_Discounted_Replacement_Cost
            + Total_Discounted_Electricity_Cost
            + Total_Discounted_Water_Cost
            - Total_Discounted_Electricity_Income
            - Total_Discounted_Hydrogen_Income
        )

        LCOH = (
            Net_Present_Cost / Total_discounted_H2
            if Total_discounted_H2 > 0
            else np.nan
        )

        LCOHfarm = (
            Net_Present_Cost / Total_discounted_H2_Consumption
            if Total_discounted_H2_Consumption > 0
            else np.nan
        )

        # -----------------------------
        # Economic results
        # -----------------------------
        economic_results = {
            "Storage threshold for sale": sell_storage_threshold,
            "No work ahead (days)": sell_free_days,
            "Hydrogen sell price (EURO/kg H2)": sell_hydrogen_price,
            "Discount Rate": Discount_Rate,
            "Battery Size (kWh)": Bsize,
            "Electrolyzer Size (kWp)": Esize,
            "Storage Size (kg H2)": Ssize,
            "PV Size (kWp)": Psize,
            "LCOH (EUR2024/kg H2)": LCOH,
            "LCOHfarm (EUR2024/kg H2)": LCOHfarm,
            "Net Present Cost (EUR2024)": Net_Present_Cost,
            "Total CAPEX (EUR2024)": Total_CAPEX,
            "PV Capex": PV_Capex,
            "Battery Capex": Battery_Capex,
            "Electrolyzer Capex": Electrolyzer_CAPEX,
            "H2 Storage Capex": H2_Storage_Capex,
            "Compressor Capex": Compressor_Capex,
            "Total Discounted Fixed OPEX (EUR2024)": Total_Discounted_OPEX_fix,
            "Annual Electrolyzer Opex": Annual_Electrolyzer_Opex,
            "Annual PV Opex": Annual_PV_Opex,
            "Annual Compressor Opex": Annual_Compressor_Opex,
            "Annual Battery Opex": Annual_Battery_Opex,
            "Annual H2 Storage Opex": Annual_H2_Storage_Opex,
            "Annual N4 Electricity Fixed Cost": Annual_N4_Electricity_Fixed_Cost,
            "Total Discounted Replacement Cost (EUR2024)": Total_Discounted_Replacement_Cost,
            "Total Discounted Electricity Cost (EUR2024)": Total_Discounted_Electricity_Cost,
            "Total Discounted Water Cost (EUR2024)": Total_Discounted_Water_Cost,
            "Total Discounted Electricity Income (EUR2024)": Total_Discounted_Electricity_Income,
            "Total Discounted Hydrogen Income (EUR2024)": Total_Discounted_Hydrogen_Income,
            "Total H2 produced (kg)": Total_H2_Produced,
            "Total H2 sold (kg)": Lifetime * Simh["Hydrogen sold (kg)"].sum() / simulation_years,
            "Number of selling events": Lifetime * Simh["Selling event"].sum() / simulation_years,
            "Total H2 deficit (kg)": Lifetime * Simh["H2def (kg H2)"].sum() / simulation_years,
            "Electrolyzer usage (%)": (
                100
                * Simh["Electrolyzer production (kWh)"].sum()
                / simulation_years
                / (8760 * Esize)
            ),
            "Electrolyzer production (kWh)": (
                Lifetime
                * Simh["Electrolyzer production (kWh)"].sum()
                / simulation_years
            ),
            "PV to Electrolyzer (kWh)": (
                Lifetime
                * Simh["PV to Electrolyzer (kWh)"].sum()
                / simulation_years
            ),
            "Grid to Electrolyzer (kWh)": (
                Lifetime
                * Simh["Grid to Electrolyzer (kWh)"].sum()
                / simulation_years
            ),
            "Consumption (kg)": (
                Lifetime
                * Simh["Consumption (kg H2)"].sum()
                / simulation_years
            ),
            "PV production (kWh)": (
                Lifetime
                * Simh["PV production (kWh)"].sum()
                / simulation_years
            ),
            "Excess electricity": (
                Lifetime
                * Simh["Excess e (kWh)"].sum()
                / simulation_years
            ),
        }

        economic_df = pd.concat(
            [economic_df, pd.DataFrame([economic_results])],
            ignore_index=True
        )


economic_df.to_csv("economic_results.csv", index=False, sep=";")